# Cargar_datos.ipynb

**Proyecto final de Machine Learning – Riesgo de pago de créditos**

En un entorno productivo este notebook **no se usaría**: los datos ya estarían materializados en una
tabla del DWH o Data Lake por otro proceso. Aquí, como el dataset es de ejemplo, se carga desde un
archivo `.csv` ubicado en la raíz del repositorio (`Base_de_datos.csv`).

**Objetivo:** cargar la base, verificar que se leyó bien y dejar claro qué contiene.


In [2]:
import sys
from pathlib import Path

import pandas as pd

# Los notebooks viven en mlops_pipeline/, así que la base está un nivel arriba.
RUTA_DATOS = Path("..") / "Base_de_datos.csv"
print("¿Existe el archivo?", RUTA_DATOS.exists())

¿Existe el archivo? True


## 1. Carga del archivo

In [3]:
def cargar_datos(ruta=RUTA_DATOS):
    """Lee el CSV y lo devuelve como DataFrame."""
    return pd.read_csv(ruta)


df = cargar_datos()
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
df.head()

Filas: 10,763 | Columnas: 23


,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,...,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,4,2025-01-07 14:40:36,1852560.0,12,32,Empleado,3500000,1000000,128650,95.227787,...,0.0,0.0,NaN,NaN,0,0,0,916148.0,Creciente,1
1,4,2025-01-09 11:18:48,3181080.0,6,34,Empleado,5000000,2000000,441817,95.227787,...,0.0,0.0,NaN,NaN,0,0,0,4473774.0,Creciente,1
2,9,2024-12-26 18:52:57,670200.0,5,34,Independiente,5000000,2000000,108632,95.227787,...,0.0,274561.0,274561.0,NaN,2,0,1,NaN,NaN,1
3,9,2024-12-04 14:20:50,506806.8,2,25,Independiente,3000000,900000,199684,95.227787,...,0.0,27564.0,27564.0,NaN,1,0,6,NaN,NaN,1
4,4,2025-04-30 18:41:57,999780.0,10,26,Empleado,2000000,600000,92509,95.227787,...,0.0,0.0,NaN,NaN,0,0,0,61000.0,Creciente,1


## 2. Verificación rápida de la carga

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10763 entries, 0 to 10762
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   tipo_credito                   10763 non-null  int64  
 1   fecha_prestamo                 10763 non-null  str    
 2   capital_prestado               10763 non-null  float64
 3   plazo_meses                    10763 non-null  int64  
 4   edad_cliente                   10763 non-null  int64  
 5   tipo_laboral                   10763 non-null  str    
 6   salario_cliente                10763 non-null  int64  
 7   total_otros_prestamos          10763 non-null  int64  
 8   cuota_pactada                  10763 non-null  int64  
 9   puntaje                        10763 non-null  float64
 10  puntaje_datacredito            10757 non-null  float64
 11  cant_creditosvigentes          10763 non-null  int64  
 12  huella_consulta                10763 non-null  int64  
 1

In [5]:
# Valores nulos por columna (se analizan a fondo en comprension_eda.ipynb)
nulos = df.isna().sum()
nulos[nulos > 0].sort_values(ascending=False).to_frame("nulos")

,nulos
tendencia_ingresos,2932
promedio_ingresos_datacredito,2930
saldo_mora_codeudor,590
saldo_principal,405
saldo_mora,156
saldo_total,156
puntaje_datacredito,6


In [6]:
# Rango de fechas de la base y balance de la variable objetivo
fechas = pd.to_datetime(df["fecha_prestamo"])
print("Primer préstamo :", fechas.min())
print("Último préstamo :", fechas.max())
print()
print("Variable objetivo (Pago_atiempo):")
print(df["Pago_atiempo"].value_counts().rename({1: "Pagó a tiempo (1)", 0: "No pagó a tiempo (0)"}))

Primer préstamo : 2024-11-26 09:17:04
Último préstamo : 2026-04-26 18:43:52

Variable objetivo (Pago_atiempo):
Pago_atiempo
Pagó a tiempo (1)       10252
No pagó a tiempo (0)      511
Name: count, dtype: int64


## 3. Comentarios

* La base tiene **10.763 créditos** y **23 columnas**: 22 variables predictoras y la variable objetivo `Pago_atiempo`.
* La clase `0` (no pagó a tiempo) es solo ~4,7 % → **problema desbalanceado**; se tendrá en cuenta al entrenar y al elegir métricas.
* Hay nulos en varias columnas (sobre todo `promedio_ingresos_datacredito` y `tendencia_ingresos`): se estudian en el EDA.
* Los datos siguen *crudos*: la limpieza y los tipos de datos se tratan en `comprension_eda.ipynb`.
